# Build KHULA JARVIS for Android

Run these cells in order in a Google Colab runtime. Upload a ZIP of the `KHULA_JARVIS` project folder (it must contain `buildozer.spec`). Buildozer compiles on Linux; this notebook produces a debug APK for testing. Do not include `.env`, API keys, `.venv`, or other secrets in the ZIP. Enter the Gemini key in the installed app.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq git zip unzip build-essential autoconf libtool pkg-config zlib1g-dev libncurses5-dev libncursesw5-dev libtinfo6 cmake libffi-dev libssl-dev libltdl-dev openjdk-17-jdk
%pip install -q buildozer cython==0.29.36

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import shutil

uploaded = files.upload()  # Select a ZIP containing the KHULA_JARVIS project folder.
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one project ZIP file.')

project_area = Path('/content/khula_upload')
if project_area.exists():
    shutil.rmtree(project_area)
project_area.mkdir(parents=True)
with ZipFile(zip_names[0]) as archive:
    archive.extractall(project_area)
spec_files = list(project_area.rglob('buildozer.spec'))
if len(spec_files) != 1:
    raise FileNotFoundError('Expected exactly one buildozer.spec in the uploaded project ZIP.')
project_root = spec_files[0].parent
print(f'Project: {project_root}')

In [ ]:
import subprocess

subprocess.run(['buildozer', '-v', 'android', 'debug'], cwd=project_root, check=True)
apk_files = sorted((project_root / 'bin').glob('*.apk'))
if not apk_files:
    raise FileNotFoundError('Buildozer finished without producing an APK in bin/.')
apk_path = apk_files[-1]
print(f'APK ready: {apk_path}')

In [ ]:
files.download(str(apk_path))